# Hierarchical makemore (WaveNet-shaped)

This is the browser compatibility fixture for the hierarchical model from lecture 5. It deliberately trains a small deterministic subset so it stays quick and editable.

In [ ]:
%pip install -q torchlite
import torch
import torch.nn.functional as F
print('torchlite', torch.__version__)

In [ ]:
words = open('data/names.txt').read().splitlines()[:64]
chars = sorted(set(''.join(open('data/names.txt').read().splitlines())))
stoi = {char: index + 1 for index, char in enumerate(chars)}
stoi['.'] = 0
block_size = 8
inputs, labels = [], []
for word in words:
    context = [0] * block_size
    for char in word + '.':
        index = stoi[char]
        inputs.append(context)
        labels.append(index)
        context = context[1:] + [index]
X, Y = torch.tensor(inputs[:128]), torch.tensor(labels[:128])
vocab_size = len(stoi)
print(X.shape, Y.shape, vocab_size)

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps, self.momentum, self.training = eps, momentum, True
        self.gamma, self.beta = torch.ones(dim), torch.zeros(dim)
        self.running_mean, self.running_var = torch.zeros(dim), torch.ones(dim)
    def __call__(self, x):
        if self.training:
            dim = 0 if x.ndim == 2 else (0, 1)
            xmean, xvar = x.mean(dim, keepdim=True), x.var(dim, keepdim=True)
        else:
            xmean, xvar = self.running_mean, self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*xmean
                self.running_var = (1-self.momentum)*self.running_var + self.momentum*xvar
        return self.out
    def parameters(self): return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x): self.out = torch.tanh(x); return self.out
    def parameters(self): return []

class Embedding:
    def __init__(self, count, dim): self.weight = torch.randn((count, dim))
    def __call__(self, indices): self.out = self.weight[indices]; return self.out
    def parameters(self): return [self.weight]

class FlattenConsecutive:
    def __init__(self, n): self.n = n
    def __call__(self, x):
        batch, time, channels = x.shape
        x = x.view(batch, time//self.n, channels*self.n)
        self.out = x.squeeze(1) if x.shape[1] == 1 else x
        return self.out
    def parameters(self): return []

class Sequential:
    def __init__(self, layers): self.layers = layers
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        self.out = x
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
torch.manual_seed(42)
n_embd, n_hidden = 8, 32
model = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size),
])
with torch.no_grad(): model.layers[-1].weight *= 0.1
parameters = model.parameters()
for parameter in parameters: parameter.requires_grad = True

losses = []
for step in range(100):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)
    losses.append(loss.item())
    for parameter in parameters: parameter.grad = None
    loss.backward()
    assert all(parameter.grad is not None for parameter in parameters)
    for parameter in parameters: parameter.data += -0.1 * parameter.grad

print(f'wavenet loss: {losses[0]:.4f} -> {losses[-1]:.4f}; parameters={sum(p.nelement() for p in parameters)}')
assert losses[-1] < losses[0] * 0.8